# File 2: Feature Selection on Weighted Early Fusion Features

**Prerequisite:** Run File 1 first (`file1_weighted_fusion.ipynb`) to generate `best_weights_per_fold.pkl`.

**Pipeline:**
1. Load the best modality weights per fold from File 1
2. For each fold, apply those weights to scale the concatenated features
3. Run the full feature selection comparison on the **weighted** features:
   - Baseline (all weighted features, no FS)
   - Mutual Information
   - RFE
   - LASSO
   - 14 EvoloPy metaheuristics

This is the same pipeline as `utd_mhad_new_val.ipynb`, but features are pre-scaled
by learned modality weights before any feature selection happens.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
import warnings
from pathlib import Path
from tqdm import tqdm
from copy import deepcopy
import random
import sys
from scipy import stats
import psutil
import tracemalloc
import gc
import json as json_lib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, classification_report)

from sklearn.feature_selection import mutual_info_classif, RFE, SelectKBest
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# EvoloPy imports
sys.path.append('../EvoloPy-master')
from EvoloPy.optimizers import BAT, CS, DE, FFA, GA, GWO, HHO, JAYA, MFO, MVO, PSO, SCA, SSA, WOA

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"PyTorch version: {torch.__version__}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
PyTorch version: 2.9.1+cu128
GPU: NVIDIA GeForce RTX 4090


## Configuration

In [3]:
# Paths
FEATURE_DIR = Path("features_new")
WEIGHTS_DIR = Path("results_weighted_fusion")  # output of File 1

RESULTS_ROOT = Path("results_weighted_fs")
RESULTS_ROOT.mkdir(exist_ok=True)
PLOTS_DIR = RESULTS_ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

# Training hyperparameters (same as original)
N_FOLDS = 5
N_EPOCHS = 300
BATCH_SIZE = 32
LEARNING_RATE = 0.001
HIDDEN_DIM = 128

# Feature selection hyperparameters
N_POPULATION = 20
MAX_ITERATIONS = 30
TARGET_FEATURE_PERCENTAGE = 0.5

# All 14 EvoloPy optimizers
EVOLOPY_OPTIMIZERS = {
    'BAT': BAT.BAT,
    'CS':  CS.CS,
    'DE':  DE.DE,
    'FFA': FFA.FFA,
    'GA':  GA.GA,
    'GWO': GWO.GWO,
    'HHO': HHO.HHO,
    'JAYA': JAYA.JAYA,
    'MFO': MFO.MFO,
    'MVO': MVO.MVO,
    'PSO': PSO.PSO,
    'SCA': SCA.SCA,
    'SSA': SSA.SSA,
    'WOA': WOA.WOA,
}

ALL_METHODS = ['baseline', 'mutual_info', 'rfe', 'lasso'] + [f'meta_{k}' for k in EVOLOPY_OPTIMIZERS.keys()]

for method in ALL_METHODS:
    (RESULTS_ROOT / method).mkdir(exist_ok=True)

print(f"Configuration:")
print(f"  N_FOLDS: {N_FOLDS}")
print(f"  N_EPOCHS: {N_EPOCHS}")
print(f"  Target feature selection: {TARGET_FEATURE_PERCENTAGE*100:.0f}%")
print(f"  Metaheuristic pop: {N_POPULATION}, iter: {MAX_ITERATIONS}")
print(f"  EvoloPy optimizers: {list(EVOLOPY_OPTIMIZERS.keys())}")
print(f"  Total methods (incl. baseline): {len(ALL_METHODS)}")

Configuration:
  N_FOLDS: 5
  N_EPOCHS: 300
  Target feature selection: 50%
  Metaheuristic pop: 20, iter: 30
  EvoloPy optimizers: ['BAT', 'CS', 'DE', 'FFA', 'GA', 'GWO', 'HHO', 'JAYA', 'MFO', 'MVO', 'PSO', 'SCA', 'SSA', 'WOA']
  Total methods (incl. baseline): 18


## Load Data & Best Weights from File 1

In [4]:
# Load features
X_feat = joblib.load(FEATURE_DIR / "X_feat.pkl")
y = np.load(FEATURE_DIR / "y.npy")
subjects = np.load(FEATURE_DIR / "subjects.npy")
le = joblib.load(FEATURE_DIR / "label_encoder.pkl")

print(f"Loaded {len(X_feat)} samples")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Number of subjects: {len(np.unique(subjects))}")

first_sample = X_feat[0]
FEATURE_DIMS = {
    'depth': first_sample['depth_feat'].shape[0],
    'sensor': first_sample['sensor_feat'].shape[0],
    'skeleton': first_sample['skeleton_feat'].shape[0]
}
MODALITY_NAMES = list(FEATURE_DIMS.keys())
MODALITY_DIMS = list(FEATURE_DIMS.values())
TOTAL_FEATURES = sum(MODALITY_DIMS)
N_MODALITIES = len(MODALITY_NAMES)

print(f"\nFeature dimensions:")
for modality, dim in FEATURE_DIMS.items():
    print(f"  {modality}: {dim}")
print(f"  TOTAL: {TOTAL_FEATURES}")

# Load best weights from File 1
best_weights_per_fold = joblib.load(WEIGHTS_DIR / "best_weights_per_fold.pkl")

print(f"\n{'=' * 60}")
print("LOADED BEST WEIGHTS PER FOLD FROM FILE 1:")
print(f"{'=' * 60}")
for fold_idx in range(N_FOLDS):
    info = best_weights_per_fold[fold_idx]
    w_str = ', '.join([f"{MODALITY_NAMES[i]}={info['weights'][i]:.3f}" for i in range(N_MODALITIES)])
    print(f"  Fold {fold_idx+1}: {info['method']:25s} | Val: {info['val_acc']*100:.2f}% | [{w_str}]")

Loaded 1165 samples
Number of classes: 22
Number of subjects: 5

Feature dimensions:
  depth: 216
  sensor: 652
  skeleton: 1645
  TOTAL: 2513

LOADED BEST WEIGHTS PER FOLD FROM FILE 1:
  Fold 1: weighted_HHO              | Val: 99.49% | [depth=0.004, sensor=0.014, skeleton=0.020]
  Fold 2: weighted_CS               | Val: 99.49% | [depth=0.047, sensor=0.529, skeleton=0.555]
  Fold 3: weighted_GWO              | Val: 99.49% | [depth=0.039, sensor=1.000, skeleton=0.895]
  Fold 4: weighted_GA               | Val: 100.00% | [depth=0.164, sensor=0.978, skeleton=0.580]
  Fold 5: weighted_SCA              | Val: 96.68% | [depth=0.016, sensor=0.499, skeleton=0.070]


## Model & Dataset (same as original)

In [5]:
class MultiModalDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class SimpleNN(nn.Module):
    """MLP for unified feature vector (same as original)"""
    def __init__(self, input_dim, num_classes):
        super().__init__()
        hidden1 = max(128, min(512, input_dim * 2))
        hidden2 = max(64, min(256, hidden1 // 2))
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden2, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(x)


print("Model and dataset defined")

Model and dataset defined


## Helper Functions

In [6]:
def prepare_unified_features(X_feat_list, feature_mask=None):
    """Concatenate all modality features into unified vector"""
    unified = []
    for sample in X_feat_list:
        feat_vector = np.concatenate([
            sample['depth_feat'],
            sample['sensor_feat'],
            sample['skeleton_feat']
        ])
        if feature_mask is not None:
            feat_vector = feat_vector[feature_mask]
        unified.append(feat_vector)
    return np.array(unified)


def apply_modality_weights(X, weights):
    """Apply per-modality scalar weights to concatenated feature matrix."""
    X_weighted = X.copy()
    start_idx = 0
    for i, dim in enumerate(MODALITY_DIMS):
        end_idx = start_idx + dim
        X_weighted[:, start_idx:end_idx] *= weights[i]
        start_idx = end_idx
    return X_weighted


def calculate_modality_retention(binary_mask, feature_dims):
    """Calculate how many features retained per modality"""
    start_idx = 0
    retention = {}
    for modality, dim in feature_dims.items():
        end_idx = start_idx + dim
        modality_mask = binary_mask[start_idx:end_idx]
        num_selected = np.sum(modality_mask)
        percentage = (num_selected / dim) * 100
        retention[modality] = {
            'selected': int(num_selected),
            'total': dim,
            'percentage': percentage
        }
        start_idx = end_idx
    return retention


def count_model_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def get_model_size_mb(model):
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / (1024 ** 2)

def get_gpu_memory_mb():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / (1024 ** 2)
    return 0.0

def get_dataset_size_mb(X):
    return X.nbytes / (1024 ** 2)


print("Helper functions defined")

Helper functions defined


## Full Training & Evaluation (same as original)

In [7]:
def train_and_evaluate_full(model, train_loader, val_loader, test_loader, num_epochs, lr, num_classes):
    """Train model and return comprehensive metrics (same as original)"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    gpu_mem_before = get_gpu_memory_mb()
    train_start = time.time()

    train_losses = []
    best_val_acc = -1.0
    best_model_state = None
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        for features, labels in train_loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        train_losses.append(epoch_loss / len(train_loader))

        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(DEVICE)
                outputs = model(features)
                preds = torch.argmax(outputs, dim=1)
                val_correct += (preds.cpu() == labels).sum().item()
                val_total += labels.size(0)
        epoch_val_acc = val_correct / val_total

        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            best_model_state = deepcopy(model.state_dict())
    
    train_time = time.time() - train_start
    gpu_mem_after = get_gpu_memory_mb()

    model.load_state_dict(best_model_state)
    
    model.eval()
    with torch.no_grad():
        val_preds, val_true = [], []
        for features, labels in val_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(labels.numpy())
        
        test_preds, test_true = [], []
        for features, labels in test_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            test_preds.extend(preds)
            test_true.extend(labels.numpy())
    
    val_acc = accuracy_score(val_true, val_preds)
    test_acc = accuracy_score(test_true, test_preds)
    
    metrics = {
        'val_acc': val_acc,
        'test_acc': test_acc,
        'test_f1_macro': f1_score(test_true, test_preds, average='macro', zero_division=0),
        'test_f1_weighted': f1_score(test_true, test_preds, average='weighted', zero_division=0),
        'test_precision_macro': precision_score(test_true, test_preds, average='macro', zero_division=0),
        'test_recall_macro': recall_score(test_true, test_preds, average='macro', zero_division=0),
        'test_preds': np.array(test_preds),
        'test_true': np.array(test_true),
        'val_preds': np.array(val_preds),
        'val_true': np.array(val_true),
        'train_time_sec': train_time,
        'gpu_mem_before_mb': gpu_mem_before,
        'gpu_mem_after_mb': gpu_mem_after,
        'gpu_mem_peak_mb': torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0,
        'model_params': count_model_parameters(model),
        'model_size_mb': get_model_size_mb(model),
        'train_losses': train_losses,
    }
    return metrics


print("Full train/eval function defined")

Full train/eval function defined


## Feature Selection Methods (same as original)

In [8]:
# ============================================================================
# Quick training for metaheuristic fitness evaluation
# ============================================================================

def train_model_quick(model, train_loader, val_loader, epochs, lr, device):
    """Quick training for fitness evaluation"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    best_val_loss = float('inf')
    best_val_acc = 0.0
    
    for epoch in range(epochs):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                val_loss += criterion(outputs, labels).item()
                val_correct += (torch.argmax(outputs, 1) == labels).sum().item()
                val_total += labels.size(0)
        
        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        if val_loss < best_val_loss:
            best_val_loss, best_val_acc = val_loss, val_acc
    
    return best_val_loss, best_val_acc


# ============================================================================
# Metaheuristic fitness function (same as original)
# ============================================================================

def create_fitness_function_evolopy(X_train, y_train, X_val, y_val, num_classes):
    """Create fitness function for EvoloPy using (1 - accuracy) + feature penalty"""
    eval_count = [0]
    
    def fitness_function(binary_mask):
        try:
            if binary_mask.dtype != bool:
                binary_mask = binary_mask > 0.5

            num_selected = np.sum(binary_mask)
            if num_selected == 0:
                return 1.0
            
            X_tr_sel = X_train[:, binary_mask]
            X_val_sel = X_val[:, binary_mask]
            
            train_dataset = MultiModalDataset(X_tr_sel, y_train)
            val_dataset = MultiModalDataset(X_val_sel, y_val)
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
            
            model = SimpleNN(X_tr_sel.shape[1], num_classes).to(DEVICE)
            
            val_loss, val_acc = train_model_quick(
                model, train_loader, val_loader,
                epochs=30, lr=1e-3, device=DEVICE
            )
            
            del model, train_dataset, val_dataset, train_loader, val_loader
            torch.cuda.empty_cache()
            
            eval_count[0] += 1
            
            alpha = 0.95
            beta = 0.05
            feature_ratio = num_selected / len(binary_mask)
            fitness = alpha * (1.0 - val_acc) + beta * feature_ratio
            return fitness
            
        except Exception as e:
            print(f"Error in fitness: {e}")
            return 1.0
        
    return fitness_function


def s_transfer(x):
    return 1 / (1 + np.exp(-10 * (x - 0.5)))


def run_evolopy_optimizer(optimizer_name, optimizer_func, X_train, y_train, X_val, y_val, num_classes):
    """Run any EvoloPy optimizer generically (same as original)"""
    print(f"    Running EvoloPy {optimizer_name}...")
    print(f"      Fitness: (1 - accuracy), Pop: {N_POPULATION}, Iter: {MAX_ITERATIONS}")
    
    fitness_func = create_fitness_function_evolopy(X_train, y_train, X_val, y_val, num_classes)
    
    start = time.time()
    solution = optimizer_func(fitness_func, 0, 1, TOTAL_FEATURES, N_POPULATION, MAX_ITERATIONS)
    exec_time = time.time() - start
    
    binary_mask = np.random.rand(TOTAL_FEATURES) < s_transfer(solution.bestIndividual)
    modality_ret = calculate_modality_retention(binary_mask, FEATURE_DIMS)
    
    results = {
        'mask': binary_mask,
        'convergence': solution.convergence.tolist() if hasattr(solution.convergence, 'tolist') else list(solution.convergence),
        'best_fitness': float(solution.convergence[-1]) if len(solution.convergence) > 0 else float('inf'),
        'execution_time': exec_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_ret,
        'method': f'meta_{optimizer_name}'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Fitness: {results['best_fitness']:.4f}, Time: {exec_time:.1f}s")
    print(f"      Modality: D={modality_ret['depth']['percentage']:.0f}%, "
          f"S={modality_ret['sensor']['percentage']:.0f}%, "
          f"Sk={modality_ret['skeleton']['percentage']:.0f}%, ")
    
    return results


# ============================================================================
# Standard FS methods (same as original)
# ============================================================================

def run_mutual_info_fs(X_train, y_train, X_val, y_val, num_classes):
    print("    Running Mutual Information...")
    start_time = time.time()
    mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
    k = int(TOTAL_FEATURES * TARGET_FEATURE_PERCENTAGE)
    top_k_indices = np.argsort(mi_scores)[::-1][:k]
    binary_mask = np.zeros(TOTAL_FEATURES, dtype=bool)
    binary_mask[top_k_indices] = True
    execution_time = time.time() - start_time
    modality_retention = calculate_modality_retention(binary_mask, FEATURE_DIMS)
    results = {
        'mask': binary_mask, 'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)), 'num_total': len(binary_mask),
        'modality_retention': modality_retention, 'mi_scores': mi_scores, 'method': 'mutual_info'
    }
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), Time: {execution_time:.1f}s")
    return results


def run_rfe_fs(X_train, y_train, X_val, y_val, num_classes):
    print("    Running RFE...")
    start_time = time.time()
    estimator = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    k = int(TOTAL_FEATURES * TARGET_FEATURE_PERCENTAGE)
    selector = RFE(estimator, n_features_to_select=k, step=50)
    selector.fit(X_train, y_train)
    binary_mask = selector.support_
    execution_time = time.time() - start_time
    modality_retention = calculate_modality_retention(binary_mask, FEATURE_DIMS)
    results = {
        'mask': binary_mask, 'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)), 'num_total': len(binary_mask),
        'modality_retention': modality_retention, 'ranking': selector.ranking_, 'method': 'rfe'
    }
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), Time: {execution_time:.1f}s")
    return results


def run_lasso_fs(X_train, y_train, X_val, y_val, num_classes):
    print("    Running LASSO...")
    start_time = time.time()
    lasso = LogisticRegression(penalty='l1', C=0.01, solver='liblinear', random_state=42, max_iter=1000)
    lasso.fit(X_train, y_train)
    coef_abs = np.abs(lasso.coef_).sum(axis=0)
    target_count = int(TOTAL_FEATURES * TARGET_FEATURE_PERCENTAGE)
    top_indices = np.argsort(coef_abs)[::-1][:target_count]
    binary_mask = np.zeros(TOTAL_FEATURES, dtype=bool)
    binary_mask[top_indices] = True
    execution_time = time.time() - start_time
    modality_retention = calculate_modality_retention(binary_mask, FEATURE_DIMS)
    results = {
        'mask': binary_mask, 'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)), 'num_total': len(binary_mask),
        'modality_retention': modality_retention, 'coefficients': coef_abs, 'method': 'lasso'
    }
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), Time: {execution_time:.1f}s")
    return results


print("All feature selection methods defined")

All feature selection methods defined


## Main Experiment Runner

Key difference from original: **before normalizing, we apply the best modality weights per fold.**

In [9]:
def run_all_methods_per_fold():
    """
    Run ALL FS methods across ALL folds on WEIGHTED features.
    For each fold, the best modality weights from File 1 are applied
    before normalization and feature selection.
    """
    print("=" * 80)
    print("FEATURE SELECTION ON WEIGHTED EARLY FUSION FEATURES")
    print(f"Methods: baseline + 3 standard + {len(EVOLOPY_OPTIMIZERS)} metaheuristics = {len(ALL_METHODS)} total")
    print(f"Folds: {N_FOLDS}")
    print("Using best modality weights from File 1 per fold")
    print("=" * 80)
    
    X_unified = prepare_unified_features(X_feat)
    num_classes = len(np.unique(y))
    
    master_results = {m: {} for m in ALL_METHODS}
    
    gkf = GroupKFold(n_splits=N_FOLDS)
    
    for fold_idx, (train_val_idx, test_idx) in enumerate(gkf.split(X_unified, y, groups=subjects)):
        print(f"\n{'#' * 80}")
        print(f"# FOLD {fold_idx + 1}/{N_FOLDS}")
        print(f"{'#' * 80}")
        
        # Get best weights for this fold
        fold_weights = best_weights_per_fold[fold_idx]['weights']
        fold_weight_method = best_weights_per_fold[fold_idx]['method']
        w_str = ', '.join([f"{MODALITY_NAMES[i]}={fold_weights[i]:.3f}" for i in range(N_MODALITIES)])
        print(f"  Applying weights from {fold_weight_method}: [{w_str}]")
        
        # Split data
        X_train_val = X_unified[train_val_idx]
        y_train_val = y[train_val_idx]
        X_test = X_unified[test_idx]
        y_test = y[test_idx]
        
        # Reserve 1 subject for validation (same as original)
        train_val_subjects = np.unique(subjects[train_val_idx])
        val_subject = train_val_subjects[-1]
        val_mask = subjects[train_val_idx] == val_subject
        train_mask = ~val_mask
        
        X_train, y_train = X_train_val[train_mask], y_train_val[train_mask]
        X_val, y_val = X_train_val[val_mask], y_train_val[val_mask]
        
        print(f"  Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
        
        # *** APPLY MODALITY WEIGHTS (from File 1) ***
        X_train = apply_modality_weights(X_train, fold_weights)
        X_val = apply_modality_weights(X_val, fold_weights)
        X_test = apply_modality_weights(X_test, fold_weights)
        
        # Normalize (AFTER weighting)
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)
        
        orig_dataset_size_mb = get_dataset_size_mb(X_train) + get_dataset_size_mb(X_val) + get_dataset_size_mb(X_test)
        
        # =================================================================
        # Run each FS method
        # =================================================================
        for method_name in ALL_METHODS:
            print(f"\n  --- {method_name.upper()} ---")
            
            fs_start_time = time.time()
            
            if method_name == 'baseline':
                feature_mask = np.ones(TOTAL_FEATURES, dtype=bool)
                fs_results = {
                    'mask': feature_mask, 'execution_time': 0,
                    'num_selected': TOTAL_FEATURES, 'num_total': TOTAL_FEATURES,
                    'method': 'baseline'
                }
            elif method_name == 'mutual_info':
                fs_results = run_mutual_info_fs(X_train, y_train, X_val, y_val, num_classes)
                feature_mask = fs_results['mask']
            elif method_name == 'rfe':
                fs_results = run_rfe_fs(X_train, y_train, X_val, y_val, num_classes)
                feature_mask = fs_results['mask']
            elif method_name == 'lasso':
                fs_results = run_lasso_fs(X_train, y_train, X_val, y_val, num_classes)
                feature_mask = fs_results['mask']
            elif method_name.startswith('meta_'):
                opt_name = method_name.replace('meta_', '')
                opt_func = EVOLOPY_OPTIMIZERS[opt_name]
                fs_results = run_evolopy_optimizer(opt_name, opt_func, X_train, y_train, X_val, y_val, num_classes)
                feature_mask = fs_results['mask']
            else:
                raise ValueError(f"Unknown method: {method_name}")
            
            fs_time = time.time() - fs_start_time
            
            # Apply feature mask
            X_train_sel = X_train[:, feature_mask]
            X_val_sel = X_val[:, feature_mask]
            X_test_sel = X_test[:, feature_mask]
            
            opt_dataset_size_mb = get_dataset_size_mb(X_train_sel) + get_dataset_size_mb(X_val_sel) + get_dataset_size_mb(X_test_sel)
            
            # Build and train model
            model = SimpleNN(X_train_sel.shape[1], num_classes).to(DEVICE)
            
            train_dataset = MultiModalDataset(X_train_sel, y_train)
            val_dataset = MultiModalDataset(X_val_sel, y_val)
            test_dataset = MultiModalDataset(X_test_sel, y_test)
            
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
            test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
            
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            
            eval_metrics = train_and_evaluate_full(
                model, train_loader, val_loader, test_loader, N_EPOCHS, LEARNING_RATE, num_classes
            )
            
            # Modality retention
            if method_name != 'baseline':
                modality_ret = fs_results.get('modality_retention', calculate_modality_retention(feature_mask, FEATURE_DIMS))
            else:
                modality_ret = {m: {'selected': d, 'total': d, 'percentage': 100.0} for m, d in FEATURE_DIMS.items()}
            
            fold_result = {
                'fold': fold_idx,
                'method': method_name,
                'num_train': len(X_train),
                'num_val': len(X_val),
                'num_test': len(X_test),
                'val_acc': eval_metrics['val_acc'],
                'test_acc': eval_metrics['test_acc'],
                'test_f1_macro': eval_metrics['test_f1_macro'],
                'test_f1_weighted': eval_metrics['test_f1_weighted'],
                'test_precision_macro': eval_metrics['test_precision_macro'],
                'test_recall_macro': eval_metrics['test_recall_macro'],
                'num_features_selected': int(np.sum(feature_mask)),
                'num_features_total': TOTAL_FEATURES,
                'feature_retention_pct': float(np.sum(feature_mask) / TOTAL_FEATURES * 100),
                'modality_retention': modality_ret,
                'feature_mask': feature_mask,
                'modality_weights_used': fold_weights,
                'weight_method_used': fold_weight_method,
                'fs_execution_time': fs_results.get('execution_time', 0),
                'train_time_sec': eval_metrics['train_time_sec'],
                'total_time_sec': fs_time + eval_metrics['train_time_sec'],
                'model_params': eval_metrics['model_params'],
                'model_size_mb': eval_metrics['model_size_mb'],
                'original_dataset_size_mb': orig_dataset_size_mb,
                'optimized_dataset_size_mb': opt_dataset_size_mb,
                'dataset_reduction_pct': float((1 - opt_dataset_size_mb / orig_dataset_size_mb) * 100) if orig_dataset_size_mb > 0 else 0,
                'gpu_mem_peak_mb': eval_metrics['gpu_mem_peak_mb'],
                'convergence': fs_results.get('convergence', None),
                'best_fitness': fs_results.get('best_fitness', None),
            }
            
            master_results[method_name][fold_idx] = fold_result
            
            # Save per-fold result
            fold_dir = RESULTS_ROOT / method_name
            fold_save = {k: v for k, v in fold_result.items() if k not in ['feature_mask']}
            fold_save_clean = {}
            for k, v in fold_save.items():
                if isinstance(v, np.ndarray):
                    fold_save_clean[k] = v.tolist()
                elif isinstance(v, (np.floating, np.integer)):
                    fold_save_clean[k] = float(v)
                elif isinstance(v, dict):
                    fold_save_clean[k] = {}
                    for kk, vv in v.items():
                        if isinstance(vv, dict):
                            fold_save_clean[k][kk] = {kkk: float(vvv) if isinstance(vvv, (np.floating, np.integer)) else vvv for kkk, vvv in vv.items()}
                        else:
                            fold_save_clean[k][kk] = float(vv) if isinstance(vv, (np.floating, np.integer)) else vv
                else:
                    fold_save_clean[k] = v
            
            with open(fold_dir / f"fold_{fold_idx+1}.json", 'w') as f:
                json_lib.dump(fold_save_clean, f, indent=2, default=str)
            np.save(fold_dir / f"fold_{fold_idx+1}_mask.npy", feature_mask)
            
            print(f"    Test Acc: {eval_metrics['test_acc']*100:.2f}%, "
                  f"F1: {eval_metrics['test_f1_macro']*100:.2f}%, "
                  f"Features: {np.sum(feature_mask)}/{TOTAL_FEATURES} "
                  f"({np.sum(feature_mask)/TOTAL_FEATURES*100:.1f}%), "
                  f"Model: {eval_metrics['model_size_mb']:.3f}MB")
            
            del model, train_dataset, val_dataset, test_dataset
            del train_loader, val_loader, test_loader
            torch.cuda.empty_cache()
            gc.collect()
    
    print(f"\n{'=' * 80}")
    print("ALL EXPERIMENTS COMPLETED")
    print(f"{'=' * 80}")
    
    return master_results


print("Main experiment runner defined")

Main experiment runner defined


## Run All Experiments

In [10]:
master_results = run_all_methods_per_fold()

FEATURE SELECTION ON WEIGHTED EARLY FUSION FEATURES
Methods: baseline + 3 standard + 14 metaheuristics = 18 total
Folds: 5
Using best modality weights from File 1 per fold

################################################################################
# FOLD 1/5
################################################################################
  Applying weights from weighted_HHO: [depth=0.004, sensor=0.014, skeleton=0.020]
  Train: 681, Val: 198, Test: 286

  --- BASELINE ---
    Test Acc: 93.71%, F1: 93.78%, Features: 2513/2513 (100.0%), Model: 5.444MB

  --- MUTUAL_INFO ---
    Running Mutual Information...
      Selected: 1256/2513 (50.0%), Time: 11.6s
    Test Acc: 97.20%, F1: 97.21%, Features: 1256/2513 (50.0%), Model: 2.989MB

  --- RFE ---
    Running RFE...
      Selected: 1256/2513 (50.0%), Time: 2.4s
    Test Acc: 95.45%, F1: 95.56%, Features: 1256/2513 (50.0%), Model: 2.989MB

  --- LASSO ---
    Running LASSO...
      Selected: 1256/2513 (50.0%), Time: 0.5s
    Test Acc: 9

## Build Per-Fold Results CSV

In [11]:
rows = []
for method_name in ALL_METHODS:
    for fold_idx in range(N_FOLDS):
        if fold_idx not in master_results[method_name]:
            continue
        r = master_results[method_name][fold_idx]
        
        row = {
            'Method': method_name,
            'Fold': fold_idx + 1,
            'Test Accuracy (%)': r['test_acc'] * 100,
            'Val Accuracy (%)': r['val_acc'] * 100,
            'F1 Macro (%)': r['test_f1_macro'] * 100,
            'F1 Weighted (%)': r['test_f1_weighted'] * 100,
            'Precision Macro (%)': r['test_precision_macro'] * 100,
            'Recall Macro (%)': r['test_recall_macro'] * 100,
            'Features Selected': r['num_features_selected'],
            'Features Total': r['num_features_total'],
            'Feature Retention (%)': r['feature_retention_pct'],
            'FS Time (s)': r['fs_execution_time'],
            'Train Time (s)': r['train_time_sec'],
            'Total Time (s)': r['total_time_sec'],
            'Model Params': r['model_params'],
            'Model Size (MB)': r['model_size_mb'],
            'Original Dataset (MB)': r['original_dataset_size_mb'],
            'Optimized Dataset (MB)': r['optimized_dataset_size_mb'],
            'Dataset Reduction (%)': r['dataset_reduction_pct'],
            'GPU Peak (MB)': r['gpu_mem_peak_mb'],
            'Weight Method Used': r['weight_method_used'],
        }
        
        # Per-modality retention
        for mod_name in FEATURE_DIMS.keys():
            mod_ret = r['modality_retention'].get(mod_name, {})
            if isinstance(mod_ret, dict):
                row[f'{mod_name}_retention (%)'] = mod_ret.get('percentage', 0)
                row[f'{mod_name}_selected'] = mod_ret.get('selected', 0)
            else:
                row[f'{mod_name}_retention (%)'] = float(mod_ret)
        
        # Per-modality weights used
        mod_w = r['modality_weights_used']
        for i, mod_name in enumerate(MODALITY_NAMES):
            row[f'w_{mod_name}'] = mod_w[i]
        
        rows.append(row)

df_all = pd.DataFrame(rows).round(4)
df_all.to_csv(RESULTS_ROOT / "all_results_per_fold.csv", index=False)
print(f"Saved: {RESULTS_ROOT / 'all_results_per_fold.csv'}")
print(f"Shape: {df_all.shape}")
print(df_all.head(20).to_string())

Saved: results_weighted_fs/all_results_per_fold.csv
Shape: (90, 30)
         Method  Fold  Test Accuracy (%)  Val Accuracy (%)  F1 Macro (%)  F1 Weighted (%)  Precision Macro (%)  Recall Macro (%)  Features Selected  Features Total  Feature Retention (%)  FS Time (s)  Train Time (s)  Total Time (s)  Model Params  Model Size (MB)  Original Dataset (MB)  Optimized Dataset (MB)  Dataset Reduction (%)  GPU Peak (MB) Weight Method Used  depth_retention (%)  depth_selected  sensor_retention (%)  sensor_selected  skeleton_retention (%)  skeleton_selected  w_depth  w_sensor  w_skeleton
0      baseline     1            93.7063           98.9899       93.7799          93.7799              94.5675           93.7063               2513            2513               100.0000       0.0000          8.1535          8.1535       1425686           5.4444                22.3362                 22.3362                 0.0000        50.2090       weighted_HHO             100.0000             216            

## Summary Table

In [12]:
summary_rows = []
for method_name in ALL_METHODS:
    method_df = df_all[df_all['Method'] == method_name]
    if len(method_df) == 0:
        continue
    
    row = {
        'Method': method_name,
        'Mean Test Acc (%)': method_df['Test Accuracy (%)'].mean(),
        'Std Test Acc (%)': method_df['Test Accuracy (%)'].std(),
        'Mean F1 Macro (%)': method_df['F1 Macro (%)'].mean(),
        'Std F1 Macro (%)': method_df['F1 Macro (%)'].std(),
        'Mean Features Selected': method_df['Features Selected'].mean(),
        'Mean Feature Retention (%)': method_df['Feature Retention (%)'].mean(),
        'Mean FS Time (s)': method_df['FS Time (s)'].mean(),
        'Mean Train Time (s)': method_df['Train Time (s)'].mean(),
        'Mean Total Time (s)': method_df['Total Time (s)'].mean(),
        'Mean Model Params': method_df['Model Params'].mean(),
        'Mean Model Size (MB)': method_df['Model Size (MB)'].mean(),
        'Mean Dataset Reduction (%)': method_df['Dataset Reduction (%)'].mean(),
        'Mean GPU Peak (MB)': method_df['GPU Peak (MB)'].mean(),
    }
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).round(2)
df_summary = df_summary.sort_values('Mean Test Acc (%)', ascending=False)
df_summary.to_csv(RESULTS_ROOT / "summary_all_methods.csv", index=False)
print("\nSUMMARY TABLE (mean across folds):")
print(df_summary.to_string(index=False))


SUMMARY TABLE (mean across folds):
     Method  Mean Test Acc (%)  Std Test Acc (%)  Mean F1 Macro (%)  Std F1 Macro (%)  Mean Features Selected  Mean Feature Retention (%)  Mean FS Time (s)  Mean Train Time (s)  Mean Total Time (s)  Mean Model Params  Mean Model Size (MB)  Mean Dataset Reduction (%)  Mean GPU Peak (MB)
mutual_info              94.95              3.76              94.74              4.03                  1256.0                       49.98             11.69                 7.04                18.73           782102.0                  2.99                       50.02               35.33
    meta_GA              94.74              0.99              94.66              0.95                  1247.8                       49.65            420.61                 7.16               427.77           777903.6                  2.97                       50.35               35.27
    meta_DE              94.55              3.81              94.24              4.40                  

## Visualizations

In [13]:
# ============================================================================
# PLOT 1: Test Accuracy heatmap (method x fold)
# ============================================================================
pivot = df_all.pivot_table(values='Test Accuracy (%)', index='Method', columns='Fold')
pivot = pivot.reindex(ALL_METHODS)

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlGnBu', ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Test Accuracy (%)'})
ax.set_title('Test Accuracy (%) per Method per Fold (Weighted Features)', fontsize=16)
ax.set_ylabel('Method')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'accuracy_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [14]:
# ============================================================================
# PLOT 2: Box plot of test accuracy
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 7))
methods_sorted = df_summary.sort_values('Mean Test Acc (%)', ascending=False)['Method'].tolist()
sns.boxplot(data=df_all, x='Method', y='Test Accuracy (%)', order=methods_sorted, ax=ax, palette='Set2')
sns.stripplot(data=df_all, x='Method', y='Test Accuracy (%)', order=methods_sorted, ax=ax,
              color='black', alpha=0.5, size=4)
ax.set_title('Test Accuracy Distribution (Weighted Features + FS)', fontsize=14)
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'accuracy_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [15]:
# ============================================================================
# PLOT 3: Feature retention bar chart
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 7))
feat_summary = df_all.groupby('Method')['Feature Retention (%)'].agg(['mean', 'std']).reindex(ALL_METHODS)
bars = ax.bar(feat_summary.index, feat_summary['mean'], yerr=feat_summary['std'], capsize=3, alpha=0.7, color='coral')
ax.axhline(y=100, color='r', linestyle='--', label='All Features (100%)')
ax.set_ylabel('Feature Retention (%)', fontsize=12)
ax.set_title('Feature Retention by Method (Weighted Features)', fontsize=14)
plt.xticks(rotation=60, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'feature_retention.png', dpi=150, bbox_inches='tight')
plt.show()

In [16]:
# ============================================================================
# PLOT 4: Accuracy vs Feature Retention scatter
# ============================================================================
fig, ax = plt.subplots(figsize=(12, 8))
for method in ALL_METHODS:
    mdf = df_all[df_all['Method'] == method]
    if len(mdf) == 0:
        continue
    ax.scatter(mdf['Feature Retention (%)'], mdf['Test Accuracy (%)'],
               label=method, s=60, alpha=0.7)
    ax.scatter(mdf['Feature Retention (%)'].mean(), mdf['Test Accuracy (%)'].mean(),
               s=150, alpha=1.0, edgecolors='black', linewidths=1.5)

ax.set_xlabel('Feature Retention (%)', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Accuracy vs Feature Retention (Weighted Features)', fontsize=14)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'accuracy_vs_retention.png', dpi=150, bbox_inches='tight')
plt.show()

In [17]:
# ============================================================================
# PLOT 5: Per-modality retention heatmap
# ============================================================================
modality_data = {}
for method in ALL_METHODS:
    modality_data[method] = {}
    for mod in FEATURE_DIMS.keys():
        col = f'{mod}_retention (%)'
        if col in df_all.columns:
            mdf = df_all[df_all['Method'] == method]
            modality_data[method][mod] = mdf[col].mean() if len(mdf) > 0 else 0

mod_df = pd.DataFrame(modality_data).T
mod_df = mod_df.reindex(ALL_METHODS)

fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(mod_df, annot=True, fmt='.1f', cmap='RdYlGn', ax=ax, linewidths=0.5,
            vmin=0, vmax=100, cbar_kws={'label': 'Retention (%)'})
ax.set_title('Per-Modality Feature Retention (Weighted Features)', fontsize=14)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'modality_retention_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [18]:
# ============================================================================
# PLOT 6: Execution time comparison
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

time_summary = df_all.groupby('Method')['FS Time (s)'].agg(['mean', 'std']).reindex(ALL_METHODS)
axes[0].bar(time_summary.index, time_summary['mean'], yerr=time_summary['std'], capsize=3, alpha=0.7, color='steelblue')
axes[0].set_ylabel('Feature Selection Time (s)', fontsize=12)
axes[0].set_title('Feature Selection Time', fontsize=14)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=60, ha='right')

total_summary = df_all.groupby('Method')['Total Time (s)'].agg(['mean', 'std']).reindex(ALL_METHODS)
axes[1].bar(total_summary.index, total_summary['mean'], yerr=total_summary['std'], capsize=3, alpha=0.7, color='coral')
axes[1].set_ylabel('Total Time (s)', fontsize=12)
axes[1].set_title('Total Time (FS + Training)', fontsize=14)
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=60, ha='right')

plt.suptitle('Execution Time Comparison (Weighted Features)', fontsize=14)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'execution_time.png', dpi=150, bbox_inches='tight')
plt.show()

In [19]:
# ============================================================================
# PLOT 7: Convergence curves for metaheuristics
# ============================================================================
meta_methods = [m for m in ALL_METHODS if m.startswith('meta_')]

if len(meta_methods) > 0:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    for method in meta_methods:
        all_conv = []
        for fold_idx in range(N_FOLDS):
            if fold_idx in master_results[method]:
                conv = master_results[method][fold_idx].get('convergence', None)
                if conv is not None and len(conv) > 0:
                    all_conv.append(conv)
        
        if len(all_conv) > 0:
            min_len = min(len(c) for c in all_conv)
            all_conv_arr = np.array([c[:min_len] for c in all_conv])
            mean_conv = np.mean(all_conv_arr, axis=0)
            ax.plot(range(1, min_len + 1), mean_conv,
                    label=method.replace('meta_', ''), linewidth=2)
    
    ax.set_xlabel('Iteration', fontsize=12)
    ax.set_ylabel('Fitness (lower is better)', fontsize=12)
    ax.set_title('FS Convergence on Weighted Features (avg across folds)', fontsize=14)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'convergence_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

## Statistical Significance Tests

In [20]:
from scipy.stats import ttest_rel, wilcoxon

print("\n" + "=" * 80)
print("STATISTICAL SIGNIFICANCE TESTS (each method vs baseline)")
print("=" * 80)

baseline_accs = []
for fold_idx in range(N_FOLDS):
    if fold_idx in master_results['baseline']:
        baseline_accs.append(master_results['baseline'][fold_idx]['test_acc'] * 100)

stat_rows = []
for method in ALL_METHODS:
    if method == 'baseline':
        continue
    
    method_accs = []
    for fold_idx in range(N_FOLDS):
        if fold_idx in master_results[method]:
            method_accs.append(master_results[method][fold_idx]['test_acc'] * 100)
    
    if len(method_accs) == len(baseline_accs) and len(method_accs) >= 2:
        try:
            t_stat, t_pval = ttest_rel(method_accs, baseline_accs)
        except:
            t_stat, t_pval = np.nan, np.nan
        try:
            w_stat, w_pval = wilcoxon(np.array(method_accs) - np.array(baseline_accs))
        except:
            w_stat, w_pval = np.nan, np.nan
        
        diff = np.mean(method_accs) - np.mean(baseline_accs)
        sig = '***' if t_pval < 0.001 else '**' if t_pval < 0.01 else '*' if t_pval < 0.05 else ''
        
        stat_rows.append({
            'Method': method,
            'Mean Diff (%)': diff,
            'Paired t p-val': t_pval,
            'Wilcoxon p-val': w_pval,
            'Sig': sig
        })
        
        print(f"  {method:20s}: diff={diff:+.2f}%, t-test p={t_pval:.4f}, "
              f"wilcoxon p={w_pval:.4f} {sig}")

df_stats = pd.DataFrame(stat_rows)
df_stats.to_csv(RESULTS_ROOT / "statistical_tests.csv", index=False)


STATISTICAL SIGNIFICANCE TESTS (each method vs baseline)
  mutual_info         : diff=+1.03%, t-test p=0.3427, wilcoxon p=0.4375 
  rfe                 : diff=+0.28%, t-test p=0.6662, wilcoxon p=0.8750 
  lasso               : diff=-2.47%, t-test p=0.0649, wilcoxon p=0.0625 
  meta_BAT            : diff=-0.75%, t-test p=0.1980, wilcoxon p=0.2500 
  meta_CS             : diff=-0.26%, t-test p=0.7440, wilcoxon p=0.8125 
  meta_DE             : diff=+0.63%, t-test p=0.5756, wilcoxon p=0.6250 
  meta_FFA            : diff=+0.50%, t-test p=0.4158, wilcoxon p=0.4375 
  meta_GA             : diff=+0.82%, t-test p=0.3838, wilcoxon p=0.3750 
  meta_GWO            : diff=-1.03%, t-test p=0.0906, wilcoxon p=0.2500 
  meta_HHO            : diff=-0.24%, t-test p=0.6259, wilcoxon p=0.6250 
  meta_JAYA           : diff=+0.41%, t-test p=0.8002, wilcoxon p=0.8125 
  meta_MFO            : diff=-0.01%, t-test p=0.9844, wilcoxon p=1.0000 
  meta_MVO            : diff=-0.62%, t-test p=0.2308, wilcoxon p=0

## Final Summary

In [21]:
print("=" * 80)
print("FILE 2 COMPLETE — FEATURE SELECTION ON WEIGHTED FEATURES")
print("=" * 80)

print(f"\nOutput directory: {RESULTS_ROOT}")
print(f"\nModality weights used (from File 1):")
for fold_idx in range(N_FOLDS):
    info = best_weights_per_fold[fold_idx]
    w_str = ', '.join([f"{MODALITY_NAMES[i]}={info['weights'][i]:.3f}" for i in range(N_MODALITIES)])
    print(f"  Fold {fold_idx+1}: {info['method']} -> [{w_str}]")

print(f"\nPer-method folders:")
for method in ALL_METHODS:
    method_dir = RESULTS_ROOT / method
    n_files = len(list(method_dir.glob('*')))
    print(f"  {method_dir}/ ({n_files} files)")

print(f"\nPlots directory: {PLOTS_DIR}")
for p in sorted(PLOTS_DIR.glob('*.png')):
    print(f"  {p.name}")

print(f"\nCSV files:")
for p in sorted(RESULTS_ROOT.glob('*.csv')):
    print(f"  {p.name}")

print(f"\n{'=' * 80}")
print("RESULTS (sorted by mean test accuracy):")
print("=" * 80)
for _, row in df_summary.iterrows():
    print(f"  {row['Method']:20s} | Acc: {row['Mean Test Acc (%)']:6.2f}% "
          f"(+/-{row['Std Test Acc (%)']:.2f}) | F1: {row['Mean F1 Macro (%)']:6.2f}% "
          f"| Feat: {row['Mean Feature Retention (%)']:.1f}%")

FILE 2 COMPLETE — FEATURE SELECTION ON WEIGHTED FEATURES

Output directory: results_weighted_fs

Modality weights used (from File 1):
  Fold 1: weighted_HHO -> [depth=0.004, sensor=0.014, skeleton=0.020]
  Fold 2: weighted_CS -> [depth=0.047, sensor=0.529, skeleton=0.555]
  Fold 3: weighted_GWO -> [depth=0.039, sensor=1.000, skeleton=0.895]
  Fold 4: weighted_GA -> [depth=0.164, sensor=0.978, skeleton=0.580]
  Fold 5: weighted_SCA -> [depth=0.016, sensor=0.499, skeleton=0.070]

Per-method folders:
  results_weighted_fs/baseline/ (10 files)
  results_weighted_fs/mutual_info/ (10 files)
  results_weighted_fs/rfe/ (10 files)
  results_weighted_fs/lasso/ (10 files)
  results_weighted_fs/meta_BAT/ (10 files)
  results_weighted_fs/meta_CS/ (10 files)
  results_weighted_fs/meta_DE/ (10 files)
  results_weighted_fs/meta_FFA/ (10 files)
  results_weighted_fs/meta_GA/ (10 files)
  results_weighted_fs/meta_GWO/ (10 files)
  results_weighted_fs/meta_HHO/ (10 files)
  results_weighted_fs/meta_JAYA/

In [22]:
# ============================================================================
# BEST METAHEURISTIC PER FOLD: Test Accuracy & Feature Retention
# ============================================================================

# Identify metaheuristic-only methods
meta_methods = [m for m in ALL_METHODS if m.startswith('meta_')]
modality_names = list(FEATURE_DIMS.keys())  # ['depth', 'sensor', 'skeleton', 'video']

print("=" * 90)
print("BEST METAHEURISTIC PER FOLD (selected by highest test accuracy)")
print("=" * 90)

best_per_fold = []  # collect per-fold best results

for fold_idx in range(N_FOLDS):
    best_method = None
    best_test_acc = -1.0
    best_result = None

    for method_name in meta_methods:
        if fold_idx in master_results[method_name]:
            r = master_results[method_name][fold_idx]
            if r['test_acc'] > best_test_acc:
                best_test_acc = r['test_acc']
                best_method = method_name
                best_result = r

    if best_result is None:
        print(f"  Fold {fold_idx + 1}: No metaheuristic results found!")
        continue

    # Extract modality-wise retention
    mod_ret = best_result['modality_retention']
    mod_retention_pct = {}
    for mod in modality_names:
        if mod in mod_ret and isinstance(mod_ret[mod], dict):
            mod_retention_pct[mod] = mod_ret[mod].get('percentage', 0.0)
        else:
            mod_retention_pct[mod] = float(mod_ret.get(mod, 0.0))

    fold_entry = {
        'Fold': fold_idx + 1,
        'Best Metaheuristic': best_method,
        'Test Accuracy (%)': best_result['test_acc'] * 100,
        'Feature Retention (%)': best_result['feature_retention_pct'],
        'Features Selected': best_result['num_features_selected'],
        'Features Total': best_result['num_features_total'],
    }
    for mod in modality_names:
        fold_entry[f'{mod} Retention (%)'] = mod_retention_pct[mod]

    best_per_fold.append(fold_entry)

    # Print per-fold info
    print(f"\n  Fold {fold_idx + 1}: Best = {best_method}")
    print(f"    Test Accuracy:      {fold_entry['Test Accuracy (%)']:.2f}%")
    print(f"    Feature Retention:  {fold_entry['Feature Retention (%)']:.2f}% "
          f"({fold_entry['Features Selected']}/{fold_entry['Features Total']})")
    mod_str = ', '.join([f"{mod}={mod_retention_pct[mod]:.1f}%" for mod in modality_names])
    print(f"    Modality Retention: {mod_str}")

# Build DataFrame
df_best_meta = pd.DataFrame(best_per_fold)

# ============================================================================
# AVERAGE ACROSS FOLDS
# ============================================================================
print("\n" + "=" * 90)
print("AVERAGE ACROSS ALL FOLDS (Best Metaheuristic per Fold)")
print("=" * 90)

avg_test_acc = df_best_meta['Test Accuracy (%)'].mean()
std_test_acc = df_best_meta['Test Accuracy (%)'].std()
avg_feat_ret = df_best_meta['Feature Retention (%)'].mean()
std_feat_ret = df_best_meta['Feature Retention (%)'].std()

print(f"\n  Average Test Accuracy:     {avg_test_acc:.2f}% ± {std_test_acc:.2f}%")
print(f"  Average Feature Retention: {avg_feat_ret:.2f}% ± {std_feat_ret:.2f}%")

print(f"\n  Average Modality-wise Feature Retention:")
for mod in modality_names:
    col = f'{mod} Retention (%)'
    avg_mod = df_best_meta[col].mean()
    std_mod = df_best_meta[col].std()
    print(f"    {mod:>10s}: {avg_mod:.2f}% ± {std_mod:.2f}%")

# ============================================================================
# SUMMARY TABLE
# ============================================================================
print("\n" + "=" * 90)
print("PER-FOLD SUMMARY TABLE")
print("=" * 90)
display_cols = ['Fold', 'Best Metaheuristic', 'Test Accuracy (%)', 'Feature Retention (%)']
display_cols += [f'{mod} Retention (%)' for mod in modality_names]
print(df_best_meta[display_cols].to_string(index=False, float_format='%.2f'))

# Save to CSV
csv_path = RESULTS_ROOT / "best_metaheuristic_per_fold.csv"
df_best_meta.to_csv(csv_path, index=False)
print(f"\nSaved to: {csv_path}")

BEST METAHEURISTIC PER FOLD (selected by highest test accuracy)

  Fold 1: Best = meta_DE
    Test Accuracy:      96.50%
    Feature Retention:  49.86% (1253/2513)
    Modality Retention: depth=50.9%, sensor=46.8%, skeleton=50.9%

  Fold 2: Best = meta_GA
    Test Accuracy:      93.36%
    Feature Retention:  49.94% (1255/2513)
    Modality Retention: depth=51.4%, sensor=50.9%, skeleton=49.4%

  Fold 3: Best = meta_DE
    Test Accuracy:      98.18%
    Feature Retention:  47.55% (1195/2513)
    Modality Retention: depth=46.8%, sensor=48.8%, skeleton=47.2%

  Fold 4: Best = meta_SSA
    Test Accuracy:      95.00%
    Feature Retention:  49.26% (1238/2513)
    Modality Retention: depth=49.5%, sensor=46.8%, skeleton=50.2%

  Fold 5: Best = meta_JAYA
    Test Accuracy:      98.48%
    Feature Retention:  49.14% (1235/2513)
    Modality Retention: depth=42.6%, sensor=51.5%, skeleton=49.1%

AVERAGE ACROSS ALL FOLDS (Best Metaheuristic per Fold)

  Average Test Accuracy:     96.31% ± 2.16%
  